# Lower Back Pain Prediction

Classifying patients as `Normal` or `Abnormal` from biomechanical spine and
pelvis measurements.

The reusable pipeline lives in [`src/analysis.py`](../src/analysis.py); this
notebook walks through it step by step. Run every cell in order from the
repository root.

> **Educational project only.** This is not a medical diagnostic tool.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.analysis import FEATURE_NAMES, RANDOM_STATE, TEST_SIZE, build_models, load_data

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

## 1. Load and clean the data

The published CSV ships generic `Col1`..`Col12` headers plus a trailing
`Unnamed: 13` column that carries free text rather than patient data.
`load_data()` drops that column and applies the documented names.

In [ ]:
df = load_data()
df.head()

In [ ]:
print(f"Rows: {len(df)}   Columns: {df.shape[1]}")
print(f"Missing values: {int(df.isna().sum().sum())}")
df.describe().T

## 2. Target distribution

The classes are moderately imbalanced.

In [ ]:
print(df["class"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="class", hue="class", palette="Set2", legend=False)
plt.title("Class distribution")
plt.xlabel("")
plt.ylabel("Patients")
plt.show()

## 3. Correlations between features

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(df[FEATURE_NAMES].corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation matrix of biomechanical features")
plt.show()

## 4. Feature distributions by class

Which measurements separate the two groups?

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 16))

for ax, feature in zip(axes.ravel(), FEATURE_NAMES):
    sns.boxplot(data=df, x="class", y=feature, hue="class",
                palette="Set2", legend=False, ax=ax)
    ax.set_title(feature.replace("_", " ").capitalize())
    ax.set_xlabel("")

plt.tight_layout()
plt.show()

## 5. Prepare the data

- `Abnormal` → 0, `Normal` → 1
- 80% train / 20% test, stratified so both splits keep the class balance
- `StandardScaler` because Logistic Regression, SVM and KNN are scale-sensitive

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[FEATURE_NAMES]
y = (df["class"] == "Normal").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 6. Train and compare five classifiers

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score)

rows, fitted = [], {}

for name, model in build_models().items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    fitted[name] = (model, y_pred)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
    })

results = pd.DataFrame(rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
results.round(4)

In [ ]:
plt.figure(figsize=(8, 5))
ordered = results.set_index("Model").loc[list(build_models())].reset_index()
plt.bar(ordered["Model"], ordered["F1-score"], color="#2e7ebb")
plt.title("Model Comparison Based on F1 Score")
plt.xlabel("Model")
plt.ylabel("F1 Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 7. Inspect the best model

The Decision Tree wins on F1-score. It is clearly better at recognising
`Abnormal` patients than `Normal` ones.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

best_name = results.loc[0, "Model"]
best_model, best_pred = fitted[best_name]

print(f"Best model: {best_name}\n")
print(classification_report(y_test, best_pred, target_names=["Abnormal", "Normal"]))

cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=["Abnormal", "Normal"]).plot(cmap="Blues")
plt.title(f"Confusion matrix - {best_name}")
plt.show()

## 8. Which measurements drive the prediction?

In [ ]:
importance = (pd.Series(best_model.feature_importances_, index=FEATURE_NAMES)
              .sort_values())

plt.figure(figsize=(8, 6))
importance.plot(kind="barh", color="#4C9F70")
plt.title(f"Feature importance - {best_name}")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

importance.sort_values(ascending=False).round(4)

## Conclusion

The Decision Tree reaches roughly **84% accuracy** and an F1-score of **0.77**
for the `Normal` class. `degree_spondylolisthesis` dominates the decision by a
wide margin.

The dataset is small (310 patients), moderately imbalanced, and never measures
pain intensity directly, so treat this as a teaching example rather than a
clinical result.